# 01 Load Raw +Tour Dataset

Purpose:
- Start PySpark on Windows.
- Load files from `Datasets/Raw`.
- Show schema, sample rows, row count, and columns.
- Create basic POI/user/theme summaries.


## 1. Configure Python path for PySpark on Windows

In [1]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python executable:")
print(sys.executable)

Python executable:
/usr/bin/python3


## 2. Start Spark

In [2]:
import os
import sys
from pyspark.sql import *


def create_spark_session(app_name="PlusTourPySpark"):

    spark = (
        SparkSession.builder
        .appName(app_name)
        .master("local[*]")
        .config("spark.hadoop.fs.defaultFS", "file:///")
        .config(
        "spark.jars.repositories",
        "https://repos.spark-packages.org/"
        )
        .config(
            "spark.jars.packages",
            "io.graphframes:graphframes-spark4_2.13:0.9.3"
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    return spark

## 3. Set project root and raw dataset path

In [3]:
import sys
import importlib
from pathlib import Path

current_path = Path.cwd()

if current_path.name.lower() == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

importlib.invalidate_caches()

from src.spark.spark_session import create_spark_session

spark = create_spark_session("PlusTourLoadRawDataset")

print("Project root:", project_root)
print("Python executable:", sys.executable)
print("Spark version:", spark.version)
print("Spark started successfully.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/31 15:09:46 WARN Utils: Your hostname, duongthai-VirtualBox, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/05/31 15:09:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/31 15:09:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Project root: /home/duongthai/Downloads/DoAnBigData/smart-tourism
Python executable: /usr/bin/python3
Spark version: 4.1.1
Spark started successfully.


## 4. Defining project and dataset path

This cell identifies the project root folder and creates the path to the raw +Tour dataset.  
Because the notebook is stored inside the `notebooks` folder, the project root is set to one level above the current working directory.  
The variable `raw_data_path` is then used by later cells to locate and load files from `Datasets/Raw`.

In [4]:
from pathlib import Path

current_path = Path.cwd()

if current_path.name.lower() == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

raw_data_path = project_root / "Datasets" / "Raw"

print("Project root:", project_root)
print("Raw data path:", raw_data_path)
print("Raw data folder exists:", raw_data_path.exists())

Project root: /home/duongthai/Downloads/DoAnBigData/smart-tourism
Raw data path: /home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw
Raw data folder exists: True


## 5. List raw dataset files

Use this to confirm Spark is reading from the correct folder.

In [5]:
if raw_data_path.exists():
    all_items = list(raw_data_path.rglob("*"))
    files = [p for p in all_items if p.is_file()]

    print("Total files found:", len(files))
    print("First 30 files:")
    for file in files[:30]:
        print(file.relative_to(project_root))
else:
    raise FileNotFoundError(f"Cannot find raw data folder: {raw_data_path}")

Total files found: 39
First 30 files:
Datasets/Raw/Barcelona/distanceMatrix.json
Datasets/Raw/Barcelona/touristsVisits.csv
Datasets/Raw/Barcelona/POIs.csv
Datasets/Raw/Toronto/distanceMatrix.json
Datasets/Raw/Toronto/touristsVisits.csv
Datasets/Raw/Toronto/POIs.csv
Datasets/Raw/Osaka/distanceMatrix.json
Datasets/Raw/Osaka/touristsVisits.csv
Datasets/Raw/Osaka/POIs.csv
Datasets/Raw/Perth/distanceMatrix.json
Datasets/Raw/Perth/touristsVisits.csv
Datasets/Raw/Perth/POIs.csv
Datasets/Raw/Glasgow/distanceMatrix.json
Datasets/Raw/Glasgow/touristsVisits.csv
Datasets/Raw/Glasgow/POIs.csv
Datasets/Raw/Edinburgh/distanceMatrix.json
Datasets/Raw/Edinburgh/touristsVisits.csv
Datasets/Raw/Edinburgh/POIs.csv
Datasets/Raw/Athens/distanceMatrix.json
Datasets/Raw/Athens/touristsVisits.csv
Datasets/Raw/Athens/POIs.csv
Datasets/Raw/NewDelhi/distanceMatrix.json
Datasets/Raw/NewDelhi/touristsVisits.csv
Datasets/Raw/NewDelhi/POIs.csv
Datasets/Raw/Budapest/distanceMatrix.json
Datasets/Raw/Budapest/touristsVi

## 6. Load raw dataset

This reads all CSV files under `Datasets/Raw`, including nested folders.

In [6]:
import pandas as pd
import json

from pathlib import Path
from pyspark.sql.functions import col

csv_files = list(raw_data_path.rglob("*.csv"))
json_files = list(raw_data_path.rglob("*.json"))

print("CSV files found:", len(csv_files))
print("JSON files found:", len(json_files))

for f in csv_files[:10]:
    print(f)

for f in json_files[:10]:
    print(f)


CSV files found: 26
JSON files found: 13
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Barcelona/touristsVisits.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Barcelona/POIs.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Toronto/touristsVisits.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Toronto/POIs.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Osaka/touristsVisits.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Osaka/POIs.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Perth/touristsVisits.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Perth/POIs.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Glasgow/touristsVisits.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Glasgow/POIs.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Barcelona/distanceMatrix.jso

In [7]:
# Load touristsVisits.csv

visit_list = []

for file in csv_files:
    if file.name == "touristsVisits.csv":

        temp_pdf = pd.read_csv(f"file://{file}")

        temp_pdf["city"] = file.parent.name

        visit_list.append(temp_pdf)

visit_pdf = pd.concat(visit_list, ignore_index=True)

visit_df = spark.createDataFrame(visit_pdf)

print("Visits dataset loaded.")
visit_df.printSchema()
visit_df.show(10, truncate=False)


Visits dataset loaded.
root
 |-- photoID: long (nullable = true)
 |-- userID: string (nullable = true)
 |-- dateTaken: long (nullable = true)
 |-- poiID: long (nullable = true)
 |-- poiTheme: string (nullable = true)
 |-- seqID: long (nullable = true)
 |-- city: string (nullable = true)



26/05/31 15:10:52 WARN TaskSetManager: Stage 0 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):                                  (0 + 1) / 1]
  File "/home/duongthai/spark-4.1.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/home/duongthai/spark-4.1.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


+-------+-------------+----------+-----+--------+-----+---------+
|photoID|userID       |dateTaken |poiID|poiTheme|seqID|city     |
+-------+-------------+----------+-----+--------+-----+---------+
|1      |100108211@N06|1501791155|29   |Museum  |1    |Barcelona|
|2      |100108211@N06|1501792349|29   |Museum  |1    |Barcelona|
|3      |100108211@N06|1501792579|29   |Museum  |1    |Barcelona|
|4      |100108211@N06|1501793533|29   |Museum  |1    |Barcelona|
|5      |100108211@N06|1501794789|29   |Museum  |1    |Barcelona|
|6      |100108211@N06|1501803723|29   |Museum  |1    |Barcelona|
|7      |100108211@N06|1501803727|29   |Museum  |1    |Barcelona|
|8      |100108211@N06|1501803955|29   |Museum  |1    |Barcelona|
|9      |100108211@N06|1501804008|29   |Museum  |1    |Barcelona|
|10     |100108211@N06|1501804091|29   |Museum  |1    |Barcelona|
+-------+-------------+----------+-----+--------+-----+---------+
only showing top 10 rows


In [8]:
# Load POIs.csv

poi_list = []

for file in csv_files:
    if file.name == "POIs.csv":

        temp_pdf = pd.read_csv(f"file://{file}")

        temp_pdf["city"] = file.parent.name

        poi_list.append(temp_pdf)

poi_pdf = pd.concat(poi_list, ignore_index=True)

poi_df = spark.createDataFrame(poi_pdf)

print("POI dataset loaded.")
poi_df.printSchema()
poi_df.show(10, truncate=False)

POI dataset loaded.
root
 |-- poiID: long (nullable = true)
 |-- poiName: string (nullable = true)
 |-- poiLat: double (nullable = true)
 |-- poiLon: double (nullable = true)
 |-- poiTheme: string (nullable = true)
 |-- city: string (nullable = true)



+-----+-------------------+----------+---------+---------+---------+
|poiID|poiName            |poiLat    |poiLon   |poiTheme |city     |
+-----+-------------------+----------+---------+---------+---------+
|1    |Camp_Nou           |41.380896 |2.1228198|Sport    |Barcelona|
|2    |Park_Guell         |41.4144948|2.1526945|Park     |Barcelona|
|3    |Placa_de_Catalunya |41.3870154|2.1700471|Shopping |Barcelona|
|4    |Casa_Batllo        |41.3916384|2.1647698|Education|Barcelona|
|5    |La_Maquinista      |41.4425713|2.200075 |Shopping |Barcelona|
|6    |Diagonal_Mar       |41.4099658|2.2166369|Shopping |Barcelona|
|7    |Maremagnum         |41.375182 |2.182867 |Shopping |Barcelona|
|8    |Aquarium_Barcelona |41.376837 |2.184338 |Zoo      |Barcelona|
|9    |L'illa_Diagonal    |41.3898016|2.1355821|Shopping |Barcelona|
|10   |Arenas_de_Barcelona|41.3763184|2.1493331|Shopping |Barcelona|
+-----+-------------------+----------+---------+---------+---------+
only showing top 10 rows


In [9]:
distance_list = []

for file in json_files:

    if "distance" in file.name.lower():

        with open(file, "r", encoding="utf-8") as f:

            data = pd.read_json(f"file://{file}")

        temp_pdf = pd.DataFrame(data)

        temp_pdf["city"] = file.parent.name

        distance_list.append(temp_pdf)

distance_pdf = pd.concat(distance_list, ignore_index=True)

distance_df = spark.createDataFrame(distance_pdf)

print("Distance Matrix dataset loaded.")
distance_df.printSchema()

distance_df.show(10, truncate=False)

Distance Matrix dataset loaded.
root
 |-- fromPOIid: long (nullable = true)
 |-- fromPOIName: string (nullable = true)
 |-- toPOIid: long (nullable = true)
 |-- toPOIName: string (nullable = true)
 |-- duration: long (nullable = true)
 |-- distance: long (nullable = true)
 |-- city: string (nullable = true)

+---------+-----------+-------+------------------------------------+--------+--------+---------+
|fromPOIid|fromPOIName|toPOIid|toPOIName                           |duration|distance|city     |
+---------+-----------+-------+------------------------------------+--------+--------+---------+
|1        |Camp_Nou   |2      |Park_Guell                          |4578    |5569    |Barcelona|
|1        |Camp_Nou   |3      |Placa_de_Catalunya                  |3759    |5142    |Barcelona|
|1        |Camp_Nou   |4      |Casa_Batllo                         |3328    |4528    |Barcelona|
|1        |Camp_Nou   |6      |Diagonal_Mar                        |6984    |9382    |Barcelona|
|1        |

Traceback (most recent call last):
  File "/home/duongthai/spark-4.1.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/home/duongthai/spark-4.1.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe
